In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path


load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json
from tracemalloc import stop

messages: list[dict[str, str]] = []

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, XML or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, an XML or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json",
        "solution_criteria": "solution"
    },
    ...additional
]
```

* Each object must include "format": one of "json", "python", "regex", or "xml" (matching the expected solution type).
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, an XML, or a regular expression.
* Focus on tasks that do not require writing much code
* Each object must include "solution_criteria": Should mention what a good solution should look like.

Please generate 4 objects.
"""

    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response=chat(messages,stop_sequences=["```"])
    print(response)
    return json.loads(response)


In [13]:
dataset = generate_dataset()

print(dataset)

with open("dataset.json", "w") as f:
    json.dump(dataset, f)


[{'task': "Create a JSON policy document that allows an IAM user to read objects from a specific S3 bucket named 'my-data-bucket'"}, {'task': 'Write a Python function that parses an AWS CloudWatch log entry and extracts the timestamp, log level, and message'}, {'task': 'Create a regular expression that validates AWS IAM role ARN format (arn:aws:iam::account-id:role/role-name)'}, {'task': 'Generate an XML CloudFormation template snippet that defines an SQS queue with a 5-minute message retention period'}]


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    fmt = test_case.get("format", "unknown")
    prompt = f"""
Please solve the following task:

{test_case["task"]}

Respond with valid {fmt} only (no extra explanation unless needed).
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [ ]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for i, test_case in enumerate(dataset, 1):
        print(f"Running test {i}/{len(dataset)}: {test_case['task'][:60]}...")
        result = run_test_case(test_case)
        results.append(result)
        print(f"  Done test {i}")
    return results

In [ ]:
dataset_path = Path(__file__).resolve().parent / "dataset.json"
with open(dataset_path, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))